# 🐻 Bear Classifier

An educational image classification application built with Python, fastai, and deep learning.

In [ ]:
from fastai.vision.all import *
import ipywidgets as widgets
from IPython.display import display, HTML


In [ ]:
!wget -O export.pkl https://github.com/aithakiabdessamad/bear_app/raw/main/export.pkl


In [ ]:
learn = load_learner('export.pkl')


In [ ]:
upload = widgets.FileUpload(
    accept='image/*',
    multiple=False
)

upload.layout = widgets.Layout(
    width='220px',
    height='50px',
    margin='10px auto 20px auto'
)
upload.style.button_color = '#333333'


In [ ]:
image_display = widgets.Image(
    width=420,
    height=300
)

image_box = widgets.Box(
    [image_display],
    layout=widgets.Layout(
        width='450px',
        height='340px',
        margin='20px auto',
        border='2px solid #dddddd',
        border_radius='12px',
        align_items='center',
        justify_content='center',
        overflow='hidden'
    )
)


In [ ]:
result_output = widgets.Output()


In [ ]:
def show_image(change):
    if not upload.value:
        image_display.value = b''
        return

    if isinstance(upload.value, dict):
        file_info = next(iter(upload.value.values()))
    else:
        file_info = upload.value[0]

    image_display.value = bytes(file_info['content'])


upload.observe(show_image, names='value')


In [ ]:
def classify_image(change):
    if not upload.value:
        return

    if isinstance(upload.value, dict):
        file_info = next(iter(upload.value.values()))
    else:
        file_info = upload.value[0]

    image_data = bytes(file_info['content'])
    img = PILImage.create(image_data)

    pred, pred_idx, probs = learn.predict(img)

    top3 = torch.topk(probs, min(3, len(probs)))
    predictions = []

    for i, prob in zip(top3.indices, top3.values):
        label = learn.dls.vocab[i]
        percentage = prob.item() * 100
        predictions.append((label, percentage))

    confidence = probs[pred_idx].item() * 100

    with result_output:
        result_output.clear_output()

        rows = ''
        for label, percentage in predictions:
            rows += f'''
            <div class="prediction-row">
                <span>{label}</span>
                <strong>{percentage:.2f}%</strong>
            </div>
            <div class="bar-container">
                <div class="bar" style="width:{percentage}%;"></div>
            </div>
            '''

        display(HTML(f'''
        <style>
        .bear-result {{
            width: 380px;
            padding: 30px;
            margin: 25px auto;
            border-radius: 20px;
            background: rgba(255, 255, 255, 0.98);
            border: 1px solid #e2e8f0;
            box-shadow: 0 15px 35px rgba(0, 0, 0, 0.12);
            text-align: center;
            font-family: 'Inter', sans-serif;
        }}
        .bear-result-title {{ color:#555; font-size:13px; font-weight:600; letter-spacing:1.5px; text-transform:uppercase; }}
        .bear-result-prediction {{ color:#222; font-size:30px; font-weight:700; margin:10px 0 5px; }}
        .bear-result-confidence {{ color:#555; font-size:16px; margin-bottom:25px; }}
        .top-title {{ color:#555; font-size:12px; font-weight:600; letter-spacing:1px; text-transform:uppercase; margin-bottom:15px; }}
        .prediction-row {{ display:flex; justify-content:space-between; margin:10px 0; font-size:14px; color:#333; }}
        .bar-container {{ width:100%; height:7px; background:#eee; border-radius:10px; overflow:hidden; margin-top:5px; }}
        .bar {{ height:100%; background:#333; border-radius:10px; }}
        </style>
        <div class="bear-result">
            <div class="bear-result-title">Prediction</div>
            <div class="bear-result-prediction">{pred}</div>
            <div class="bear-result-confidence">Confidence: <strong>{confidence:.2f}%</strong></div>
            <div class="top-title">Top 3 Predictions</div>
            {rows}
        </div>
        '''))


In [ ]:
classify_button = widgets.Button(
    description='🔍 CLASSIFY',
    layout=widgets.Layout(
        width='190px',
        height='52px',
        margin='15px auto'
    )
)
classify_button.style.button_color = '#2563eb'
classify_button.on_click(classify_image)


In [ ]:
title = widgets.HTML('''
<div class="bear-title" style="text-align:center; padding:10px 0 20px 0; font-family:'Inter', sans-serif;">
    <h1 style="color:#1f2937; font-size:40px; font-weight:800; letter-spacing:-1px; margin:0 0 8px 0;">🐻 Bear Classifier</h1>
    <p style="color:#6b7280; font-size:15px; font-weight:400; letter-spacing:0.2px; margin:0;">Upload an image and let the AI identify the type of bear.</p>
</div>
''')


In [ ]:
educational_info = widgets.HTML('''
<div class="educational-project" style="width:600px; margin:35px auto 10px auto; padding:25px 20px; border-top:1px solid #d5dce3; text-align:center; font-family:'Inter', sans-serif;">
    <h3 style="color:#1f2937; font-size:18px; font-weight:700; margin:0 0 12px 0;">Educational Project</h3>
    <p style="color:#4b5563; font-size:14px; line-height:1.7; margin:0 auto; max-width:560px;">
        This project was developed by me for educational purposes. It is designed for Master's and Engineering students who want to develop their skills in <strong>Deep Learning</strong>, <strong>Machine Learning</strong>, and <strong>Python programming</strong>.
    </p>
    <p style="color:#6b7280; font-size:12px; margin-top:15px; margin-bottom:0;">Built with Python &amp; Deep Learning</p>
</div>
''')


In [ ]:
app = widgets.VBox(
    [title, upload, image_box, classify_button, result_output, educational_info],
    layout=widgets.Layout(
        width='100%',
        align_items='center',
        padding='20px'
    )
)

app_card = widgets.Box(
    [app],
    layout=widgets.Layout(
        width='700px',
        margin='30px auto',
        padding='30px',
        border='1px solid #d5dee7',
        border_radius='24px'
    )
)

app_card.add_class('app-card')
display(app_card)


In [ ]:
display(HTML('''
<style>
body { background:#eef5f9 !important; }
.app-card { background:#ffffff !important; box-shadow:0 20px 50px rgba(0,0,0,0.12) !important; }
</style>
'''))
